# 1. AIND Ephys Preprocessing

Build an `AINDEPhysPreprocessingScanConfig`, expand it with `GridScanGenerationTask`, and run `AINDEPhysPreprocessingTask` for each coordinate. The task first splits the NWB file into jobs and writes them as `job_<i>.json` into its output folder. For every job it then applies the AIND preprocessing chain with spikeinterface (phase shift when the probe needs it, highpass filter, bad-channel detection and removal, common median reference), saves the result to `preprocessed_<name>/`, and estimates the probe's motion into `motion_<name>/`.

The toy recording is shorter than the pipeline's 120 s minimum, so `min_preprocessing_duration` is lowered, and it is clipped to 9 s.

## Imports

In [1]:
import json
from pathlib import Path

from obi_one.core.info import Info
from obi_one.core.run_tasks import run_tasks_for_generated_scan
from obi_one.core.scan_generation import GridScanGenerationTask
from obi_one.scientific.tasks.aind_ephys._01_preprocessing.config import (
    AINDEPhysPreprocessingScanConfig,
)
from obi_one.scientific.tasks.aind_ephys.blocks import JobKwargs

## Build the scan config

The input is notebook 00's NWB file. On the platform, `recording=ExtracellularRecordingFromID(id_str=...)` takes the place of `nwb_path`, and the task downloads the recording's NWB asset. The `dispatch` block holds the dispatch settings.

In [2]:
obi_output = (Path.cwd() / "../../../../../../obi-output").resolve()
nwb_path = obi_output / "00_toy_example_recording" / "toy_recording.nwb"
assert nwb_path.exists(), f"{nwb_path} not found. Run notebook 00 first."

scan_config = AINDEPhysPreprocessingScanConfig(
    info=Info(
        campaign_name="Toy preprocessing",
        campaign_description="Toy preprocessing of the toy recording.",
    ),
    initialize=AINDEPhysPreprocessingScanConfig.Initialize(
        nwb_path=nwb_path,
        min_preprocessing_duration=0.0,
        t_start=0.0,
        t_stop=9.0,
    ),
    job_kwargs=JobKwargs(n_jobs=1),
)

## Generate the grid scan and run the preprocessing task

In [3]:
grid_scan = GridScanGenerationTask(
    form=scan_config,
    output_root=obi_output / "01_aind_ephys_preprocessing" / "grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

run_tasks_for_generated_scan(grid_scan)

/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-one/.venv/lib/python3.12/site-packages/spikeinterface/core/baserecordingsnippets.py:110: FutureWarning: The 'in_place' argument is deprecated and will be removed in version 0.106.0. The `set_probe/probegroup()` are now always in place; please remove the in_place argument.
  return self.set_probegroup(probegroup, group_mode=group_mode, in_place=in_place)


[PosixPath('/Users/james/Documents/obi/code/obi-main/worktree-groups/spike_sorting_pipeline/obi-output/01_aind_ephys_preprocessing/grid_scan/0')]

## Load the preprocessed recording and the bad-channel labels

In [4]:
from collections import Counter

import spikeinterface as si

coord_dir = Path(grid_scan.single_configs[0].coordinate_output_root)
for binary_file in sorted(coord_dir.glob("binary_*.json")):
    name = binary_file.stem.removeprefix("binary_")
    recording = si.load(binary_file, base_folder=coord_dir)
    record = json.loads((coord_dir / f"data_process_preprocessing_{name}.json").read_text())
    print(name)
    print(" ", recording)
    print("  steps:", list(record["parameters"]["steps"]))
    print("  channel labels:", dict(Counter(record["outputs"]["channel_labels"])))
    print("  motion estimated:", (coord_dir / f"motion_{name}").is_dir())

block0_acquisition-ElectricalSeriesRaw_recording1
  BinaryFolderRecording: 31 channels - 25.0kHz - 1 segments - 225,001 samples - 9.00s 
                       float32 dtype - 26.61 MiB
  steps: ['highpass_filter', 'detect_and_remove_bad_channels', 'common_reference']
  channel labels: {'good': 31, 'dead': 1}
  motion estimated: True
